# 01 — Domain-Adaptive Pre-Training (DAPT)

## What DAPT means and when to use it

**Domain-Adaptive Pre-Training (DAPT)** continues training an existing language model on material from a particular domain, such as robotics manuals.

**Continued pre-training (CPT)** is the broader activity of continuing a pre-training objective on additional data. DAPT is its domain-focused form. It typically keeps causal next-token prediction but starts with existing weights rather than random initialization.

**What problem does it solve?** When documents are plentiful but example conversations are scarce, DAPT can adapt vocabulary usage, style, and contextual patterns. It does not itself specify an assistant's response format, guarantee factual recall, or provide document retrieval.

A **Base checkpoint** primarily reflects pre-training; a **post-trained checkpoint** has additionally learned behaviors such as instruction following. This project's real DAPT profile starts from Base weights.


## NovaBot: supervision inside ordinary text

**Fictional explanatory sample:**

~~~json
{"text":"NovaBot has three modes: idle, mapping, navigation."}
~~~

No human labels the “correct next word”; the document supplies the targets:

| Prefix | Next illustrative target |
|---|---|
| NovaBot | has |
| NovaBot has | three |
| NovaBot has three | modes |

This table uses words for intuition; real token boundaries can differ. **Self-supervised learning** derives targets from the data itself. Raw text can therefore train a model without question/answer annotations.

The intended change is a greater probability for domain-appropriate continuations. It does not follow that every future question about NovaBot will be answered correctly.


## How the objective works

A **causal language model** predicts using earlier positions, not later ones. Let $x_t$ be token $t$, $x_{<t}$ its preceding context, and $\theta$ the trainable parameters. The model assigns probability $p_\theta(x_t\mid x_{<t})$ to the observed token:

$$L=-\frac{1}{N}\sum_{t\in V}\log p_\theta(x_t\mid x_{<t}).$$

$V$ is the set of valid prediction positions, $N=|V|$ counts them, and $\log$ is the natural logarithm. This **negative log-likelihood** is implemented as next-token **cross-entropy**: low probability for the observed target means a larger penalty.

**Hand-constructed calculation:** probability 0.8 for “three” gives $-\log(0.8)\approx0.223$; probability 0.2 gives about 1.609. These are teaching numbers, not measured model outputs.

The model aligns the output at position $t$ with the target at $t+1$. Padding is ignored; real end-of-sequence (EOS) tokens remain targets.


## Connection to this notebook

paragraphs reads domain text, splits separates documents, and collate_text() produces input_ids, attention_mask and labels. DAPT scores all real next-token positions; supervised answer training usually excludes prompt targets.

**Packing** concatenates short sequences into longer blocks. An EOS marker indicates a boundary but does not impose an attention barrier. The displayed packed_blocks allow cross-document context; the actual explicit loop keeps documents separate.

outputs.loss computes the objective, loss.backward() computes gradients, and optimizer.step() updates parameters. **Low-Rank Adaptation (LoRA)** learns small additions to frozen weights; **Quantized Low-Rank Adaptation (QLoRA)** also compresses the frozen base. Either strategy can implement DAPT.

**Perplexity** is the exponential of average token loss under the same tokenizer and mask. **Catastrophic forgetting** means domain adaptation harms previous capabilities, so lower domain loss alone is insufficient evidence of improvement. The runnable domain fixtures remain unchanged; NovaBot is an explanatory example.


## Common confusions and quick check

DAPT and **Supervised Fine-Tuning (SFT)** can both use cross-entropy. SFT learns demonstrated responses; DAPT learns from domain documents. Their distinction is not necessarily a different mathematical loss.

1. Where do targets come from if you only have manuals?
2. Does an EOS token isolate two packed documents?

<details>
<summary>Answers</summary>

1. From successive tokens in the manuals themselves. Human task labels are unnecessary.
2. No. Explicit attention isolation is needed for that guarantee. The notebook distinguishes simple concatenation from training on separate documents.

</details>


## Before running the experiment

**Learning goals:** load domain documents, preserve boundaries, compare packing choices, implement causal training, and measure held-out token loss. Use a Base checkpoint for a real run.

**Prerequisites:** basic Python. The concepts needed for this lesson are introduced above. Run every cell in order in a fresh kernel. The default `tiny_cpu` mode has random weights and a synthetic vocabulary: output quality is not evidence of Qwen's capabilities. Use `local_pretrained` for an already downloaded checkpoint.

[Course index](README.md) · [Execution and data flow](../docs/EXECUTION_AND_DATA_FLOW.md)

**Experiment contract:** inspect inputs before training, keep held-out records separate, check gradients/parameter changes, then save and reload. Each notebook is independent.

[Terminology reference](../docs/GLOSSARY.md) · [Compare training methods](../docs/TRAINING_METHODS.md)


## Choose the experiment

The model directory must contain its own weights, tokenizer, processor and chat template. The environment variables below are optional; edit the parameter cell directly in Jupyter. Files created by this lesson stay under its output directory.


In [ ]:
LESSON = "01"
# Parameters: change these before running the notebook from top to bottom.
import csv
import json
import math
import os
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor

from finetunelab.education import (
    inspect_local_checkpoint,
    make_tiny_checkpoint,
    project_root,
    token_table,
)
from finetunelab.tuning import parameter_report

MODE = os.environ.get("FTLAB_NOTEBOOK_MODE", "tiny_cpu")
LOCAL_MODEL_PATH = Path(os.environ.get("FTLAB_LOCAL_MODEL", "models/Qwen3.5-2B"))
LOCAL_TEACHER_PATH = Path(os.environ.get("FTLAB_LOCAL_TEACHER", "models/Qwen3.5-4B"))
ROOT = project_root()
DATA_ROOT = Path(os.environ.get("FTLAB_LESSON_DATA", str(ROOT / "examples/education")))
OUTPUT_ROOT = Path(os.environ.get("FTLAB_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebooks")))
OUTPUT = OUTPUT_ROOT / LESSON
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
assert MODE in {"tiny_cpu", "local_pretrained"}
DEVICE = torch.device("cpu" if MODE == "tiny_cpu" else "cuda")
if MODE == "local_pretrained" and not torch.cuda.is_available():
    raise RuntimeError(
        "The local_pretrained teaching profile requires a CUDA PyTorch installation."
    )
DTYPE = torch.float32 if MODE == "tiny_cpu" else torch.bfloat16
MODEL_PATH = (
    make_tiny_checkpoint(OUTPUT / "initial", seed=SEED)
    if MODE == "tiny_cpu"
    else LOCAL_MODEL_PATH.expanduser().resolve()
)
checkpoint_info = inspect_local_checkpoint(MODEL_PATH)
if MODE == "local_pretrained":
    original_base = os.environ.get("FTLAB_BASE_CHECKPOINT") or checkpoint_info["config"].get(
        "finetunelab_base_checkpoint",
        checkpoint_info["config"].get("_name_or_path", MODEL_PATH.name),
    )
    if original_base not in {
        "Qwen/Qwen3.5-2B-Base",
        "Qwen/Qwen3.5-4B-Base",
        "Qwen3.5-2B-Base",
        "Qwen3.5-4B-Base",
    }:
        raise ValueError(
            "DAPT needs a Base snapshot. Set FTLAB_BASE_CHECKPOINT to its original repository ID."
        )
print({"mode": MODE, "device": str(DEVICE), "checkpoint": str(MODEL_PATH)})
print(checkpoint_info["files"])

## Load the local checkpoint

`from_pretrained()` constructs the official PyTorch modules and fills their tensors from the checkpoint. `local_files_only=True` prevents a missing local file from becoming a network download. BF16 and NF4 serve different roles: computation precision versus storage of the frozen base.


In [ ]:
# A checkpoint includes both weights and the preprocessing contract.
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
load_kwargs = {
    "local_files_only": True,
    "dtype": DTYPE,
    "attn_implementation": "eager" if MODE == "tiny_cpu" else "sdpa",
}
# Real 2B teaching runs default to QLoRA. CPU fixtures use ordinary LoRA.
USE_QLORA = MODE == "local_pretrained" and LESSON not in {"00a", "00b", "04"}
if USE_QLORA:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    load_kwargs["device_map"] = {"": torch.cuda.current_device()}
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
if not USE_QLORA:
    model.to(DEVICE)
model.config.use_cache = False
print(type(model).__name__, parameter_report(model))

## From local Q&A to train/validation/test records

A dataset row is not yet a tensor. Preserve the original group identity so examples from one conversation stay together. These tiny held-out splits demonstrate plumbing; use representative, larger splits in real experiments.


In [ ]:
# Convert local Q&A rows to canonical conversations; preserve provenance.
QA_FILE = Path(os.environ.get("FTLAB_QA_FILE", str(DATA_ROOT / "qa.csv")))
if QA_FILE.suffix.lower() == ".csv":
    with QA_FILE.open(encoding="utf-8", newline="") as handle:
        raw_rows = list(csv.DictReader(handle))
elif QA_FILE.suffix.lower() == ".jsonl":
    raw_rows = [
        json.loads(line)
        for line in QA_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    raise ValueError("This converter accepts CSV or JSONL Q&A files.")
for row in raw_rows:
    if (
        not row.get("group_id")
        or not row.get("question", "").strip()
        or not row.get("answer", "").strip()
    ):
        raise ValueError("Every Q&A needs a group_id, nonempty question, and nonempty answer.")
    row.setdefault("rejected", "")
records = [
    {
        "group_id": row["group_id"],
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "prompt": row["question"],
        "chosen": row["answer"],
        "rejected": row["rejected"],
    }
    for row in raw_rows
]


def split_records(rows, seed=SEED):
    # Deduplicate before splitting. Never split one document/conversation group.
    unique = {}
    for row in rows:
        key = json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)
        unique.setdefault(key, row)
    groups = sorted({row["group_id"] for row in unique.values()})
    if len(groups) < 3:
        raise ValueError("Provide at least three independent document/conversation groups.")
    random.Random(seed).shuffle(groups)
    validation_groups, test_groups = set(groups[:1]), set(groups[1:2])
    splits = {"train": [], "validation": [], "test": []}
    for row in unique.values():
        split = (
            "validation"
            if row["group_id"] in validation_groups
            else "test"
            if row["group_id"] in test_groups
            else "train"
        )
        splits[split].append(row)
    return splits


splits = split_records(records)
for split, rows in splits.items():
    with (OUTPUT / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            canonical = {"group_id": row["group_id"], "messages": row["messages"]}
            handle.write(json.dumps(canonical, ensure_ascii=False) + "\n")
print({split: len(rows) for split, rows in splits.items()})
print("Raw:", raw_rows[0])
print("Canonical:", records[0])

## Chat rendering, tokenization and loss masking

The chat template supplies role delimiters. Attention masks describe real positions versus padding; labels choose prediction targets. A user token can be visible to attention while its label is `-100`. Assistant EOS should remain supervised even if its ID equals the padding ID.


In [ ]:
def render(messages, generation=False):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=generation,
        enable_thinking=False,
    )


def encode_conversation(messages):
    # Template-provided generation masks are preferred. They include assistant EOS.
    template = tokenizer.chat_template or ""
    if "generation" in template:
        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        supervised = encoded["assistant_masks"]
    else:
        # For templates without generation annotations, verify prefix alignment.
        # Do not guess a token count by separately tokenizing the answer.
        ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
        supervised = [0] * len(ids)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            prefix = tokenizer(render(messages[:index], generation=True), add_special_tokens=False)[
                "input_ids"
            ]
            completed = tokenizer(render(messages[: index + 1]), add_special_tokens=False)[
                "input_ids"
            ]
            if ids[: len(prefix)] != prefix or ids[: len(completed)] != completed:
                raise ValueError(
                    "Template is not prefix-stable; use a training template with generation tags."
                )
            supervised[len(prefix) : len(completed)] = [1] * (len(completed) - len(prefix))
    if not any(supervised[1:]):
        raise ValueError("No assistant target tokens remain.")
    return {
        "input_ids": ids,
        "labels": [t if keep else -100 for t, keep in zip(ids, supervised, strict=False)],
    }


def collate_text(rows):
    items = [encode_conversation(row["messages"]) for row in rows]
    encoded = tokenizer.pad(
        [{"input_ids": item["input_ids"]} for item in items],
        padding=True,
        return_tensors="pt",
    )
    # Padding labels are independent of the pad token ID (pad may equal EOS).
    labels = torch.full_like(encoded["input_ids"], -100)
    for index, item in enumerate(items):
        labels[index, : len(item["labels"])] = torch.tensor(item["labels"])
    encoded["labels"] = labels
    return dict(encoded)


batch = collate_text(splits["train"][:2])
print(render(splits["train"][0]["messages"]))
print({name: tuple(value.shape) for name, value in batch.items()})
display(token_table(tokenizer, batch))

## Replace Q&A with domain documents

DAPT supervises every non-padding next token. Add EOS to mark document boundaries. Packing can improve utilization, but an EOS alone does not prevent cross-document attention; the simple concatenation example below intentionally permits it. Split documents before packing.


In [ ]:
paragraphs = [
    p.strip()
    for p in (DATA_ROOT / "domain.txt").read_text(encoding="utf-8").split("\n\n")
    if p.strip()
]
splits = {
    "train": [{"text": text} for text in paragraphs[:-2]],
    "validation": [{"text": paragraphs[-2]}],
    "test": [{"text": paragraphs[-1]}],
}


def collate_text(rows):
    sequences = [
        tokenizer(row["text"], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
        for row in rows
    ]
    encoded = tokenizer.pad(
        [{"input_ids": ids} for ids in sequences], padding=True, return_tensors="pt"
    )
    labels = encoded["input_ids"].clone()
    labels[encoded["attention_mask"] == 0] = -100
    encoded["labels"] = labels
    return dict(encoded)


stream = []
boundaries = []
for row in splits["train"]:
    stream.extend(tokenizer(row["text"], add_special_tokens=False)["input_ids"])
    stream.append(tokenizer.eos_token_id)
    boundaries.append(len(stream))
packed_blocks = [stream[i : i + 16] for i in range(0, len(stream), 16)]
print({"document_ends": boundaries, "packed_lengths": list(map(len, packed_blocks))})
display(token_table(tokenizer, collate_text(splits["train"][:2])))
# The training loop below keeps documents separate for clarity.
if MODE == "local_pretrained":
    original_id = checkpoint_info["config"].get("_name_or_path", str(MODEL_PATH))
    print("Confirm this snapshot's Base provenance before a real DAPT run:", original_id)

## Select parameters that may change

LoRA freezes the pretrained matrices and adds low-rank updates. Its zero-initialized B matrices mean some A gradients may be zero on the first step; at least one adapter must change. The frozen vision backbone is excluded. Set `TUNING` to `full` or `selective` to compare on the tiny model; full tuning of a real model needs substantially more memory.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# This is the same choice represented by tuning.strategy in a framework YAML.
TUNING = "lora"
if USE_QLORA and TUNING != "lora":
    raise ValueError("Full/selective tuning requires reloading with USE_QLORA=False.")
if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()
if TUNING == "lora":
    model = get_peft_model(
        model,
        LoraConfig(
            r=4 if MODE == "tiny_cpu" else 16,
            lora_alpha=8 if MODE == "tiny_cpu" else 32,
            target_modules="all-linear",
            exclude_modules=r".*(?:visual|vision).*",
            task_type="CAUSAL_LM",
            lora_dropout=0.0,
        ),
    )
elif TUNING == "selective":
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("norm" in name or "lm_head" in name) and "visual" not in name
elif TUNING != "full":
    raise ValueError(TUNING)
trainable = [p for p in model.parameters() if p.requires_grad]
print(parameter_report(model))
# A small parameter sample avoids copying a 2B/4B model just to audit updates.
before = {name: p.detach().flatten()[:32].cpu().clone() for name, p in model.named_parameters()}
frozen_names = {name for name, p in model.named_parameters() if not p.requires_grad}

## Establish a baseline

Use `eval()` plus `no_grad()` for measurement, and switch back to `train()` for updates. We aggregate causal loss by supervised token count. Greedy decoding makes the before/after and reload comparisons reproducible on the same device.


In [ ]:
def to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def precision_context():
    return torch.autocast("cuda", dtype=torch.bfloat16) if DEVICE.type == "cuda" else nullcontext()


def validation_loss(current_model, rows, collator=collate_text):
    current_model.eval()
    weighted_loss, target_count = 0.0, 0
    with torch.no_grad(), precision_context():
        for row in rows:
            encoded = to_device(collator([row]))
            count = int((encoded["labels"][:, 1:] != -100).sum())
            loss = current_model(**encoded, use_cache=False).loss
            weighted_loss += float(loss) * count
            target_count += count
    return weighted_loss / max(target_count, 1)


def generate_answer(current_model, prompt="What is the color of sky ?"):
    current_model.eval()
    text = render([{"role": "user", "content": prompt}], generation=True)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt")
    inputs.pop("token_type_ids", None)
    with torch.no_grad(), precision_context():
        tokens = current_model.generate(
            **to_device(dict(inputs)),
            max_new_tokens=4 if MODE == "tiny_cpu" else 32,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokens[:, inputs["input_ids"].shape[1] :].cpu()


baseline_loss = validation_loss(model, splits["validation"])
baseline_answer = generate_answer(model)
print(
    {
        "baseline_validation_loss": baseline_loss,
        "baseline_answer": tokenizer.decode(baseline_answer[0], skip_special_tokens=True),
    }
)

## Objective and perplexity

The objective is the mean negative log-probability of domain tokens. `exp(loss)` gives perplexity for this exact tokenizer and masking scheme; it is not directly comparable across arbitrary tokenizers. DAPT may erode instruction-following behavior, so retain separate downstream evaluations.


In [ ]:
model.eval()
example = to_device(collate_text(splits["train"][:2]))
with torch.no_grad():
    result = model(**example, use_cache=False)
    manual_ce = F.cross_entropy(
        result.logits[:, :-1].float().reshape(-1, result.logits.shape[-1]),
        example["labels"][:, 1:].reshape(-1),
        ignore_index=-100,
    )
torch.testing.assert_close(manual_ce, result.loss.float(), rtol=1e-4, atol=1e-4)
print("Baseline perplexity:", math.exp(baseline_loss))

## Explicit document training loop

This uses the same optimizer mechanics as SFT but a different label mask.


In [ ]:
# The loss is averaged over valid target tokens, not padded positions.
train_loader = DataLoader(splits["train"][:5], batch_size=2, shuffle=False, collate_fn=collate_text)
ACCUMULATION = 2
EPOCHS = 1 if MODE == "tiny_cpu" else 2
optimizer = torch.optim.AdamW(trainable, lr=2e-3 if MODE == "tiny_cpu" else 2e-4)
updates_per_epoch = math.ceil(len(train_loader) / ACCUMULATION)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda step: max(0.1, 1 - step / (EPOCHS * updates_per_epoch)),
)
history = []
optimizer.zero_grad(set_to_none=True)
for _epoch in range(EPOCHS):
    batches = list(train_loader)  # Tiny lesson only; stream windows for large corpora.
    for window_start in range(0, len(batches), ACCUMULATION):
        window = batches[window_start : window_start + ACCUMULATION]
        counts = [int((item["labels"][:, 1:] != -100).sum()) for item in window]
        total_targets = sum(counts)
        if total_targets == 0:
            raise ValueError("This accumulation window has no supervised targets.")
        model.train()
        for batch, count in zip(window, counts, strict=False):
            with precision_context():
                outputs = model(**to_device(batch), use_cache=False)
                # Correct even for the final short window and unequal token lengths.
                loss = outputs.loss * count / total_targets
            assert torch.isfinite(loss)
            loss.backward()
            history.append(float(outputs.loss.detach()))
        assert all(torch.isfinite(p.grad).all() for p in trainable if p.grad is not None)
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        print({"update": scheduler.last_epoch, "loss": history[-1], "grad_norm": float(grad_norm)})
assert scheduler.last_epoch == EPOCHS * updates_per_epoch

## Inspect updates and held-out behavior

A finite loss and a changed adapter prove an update occurred, not that a model became useful. Inspect validation loss and example generations together. Keep the test set out of hyperparameter selection.


In [ ]:
changed = []
for name, parameter in model.named_parameters():
    same = torch.equal(before[name], parameter.detach().flatten()[:32].cpu())
    if name in frozen_names:
        assert same, f"Frozen parameter changed: {name}"
    elif not same:
        changed.append(name)
assert changed, "No trainable parameter sample changed."
after_loss = validation_loss(model, splits["validation"])
after_answer = generate_answer(model)
test_loss = validation_loss(model, splits["test"])
print(
    {
        "baseline_loss": baseline_loss,
        "after_loss": after_loss,
        "test_loss": test_loss,
        "changed_parameter_samples": changed[:5],
    }
)
print("Before:", tokenizer.decode(baseline_answer[0], skip_special_tokens=True))
print("After: ", tokenizer.decode(after_answer[0], skip_special_tokens=True))
# Do not assert that generalization improves after one synthetic update.
assert math.isfinite(after_loss) and math.isfinite(test_loss)

## Save, reload, and verify

`save_pretrained()` saves inference artifacts; it does not save the optimizer or training position. A PEFT artifact needs its original base. This cell creates a separate model object and compares generated token IDs. Lesson 08 covers exact resume and merging.


In [ ]:
from peft import PeftModel

artifact = OUTPUT / "final"
model.save_pretrained(artifact, safe_serialization=True)
processor.save_pretrained(artifact)
expected_tokens = generate_answer(model)
reloaded_processor = AutoProcessor.from_pretrained(artifact, local_files_only=True)
assert reloaded_processor.tokenizer.get_vocab() == tokenizer.get_vocab()
assert reloaded_processor.chat_template == processor.chat_template
processor = reloaded_processor
tokenizer = processor.tokenizer
# Reload independently, rather than reusing the trained Python object.
if (artifact / "adapter_config.json").exists():
    reload_base = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
    if not USE_QLORA:
        reload_base.to(DEVICE)
    reloaded = PeftModel.from_pretrained(reload_base, artifact, local_files_only=True)
else:
    reloaded = AutoModelForImageTextToText.from_pretrained(artifact, **load_kwargs)
    if not USE_QLORA:
        reloaded.to(DEVICE)
actual_tokens = generate_answer(reloaded)
assert torch.equal(expected_tokens, actual_tokens), "Greedy outputs changed after reload."
report = {
    "mode": MODE,
    "base_checkpoint": str(MODEL_PATH),
    "artifact": str(artifact),
    "baseline_validation_loss": baseline_loss,
    "validation_loss": after_loss,
    "test_loss": test_loss,
    "reload_tokens_equal": True,
}
(OUTPUT / "lesson_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
del reloaded
if "reload_base" in globals():
    del reload_base
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Interpretation, common failures, and exercises

- If loss is NaN, inspect the number of supervised targets, precision and learning rate before adding steps.
- If every label is `-100`, repair the template/mask; an empty objective cannot teach anything.
- If frozen parameters change, inspect the trainable parameter report and optimizer parameter list.
- If tiny generations look meaningless, that is expected from random initialization and a tiny vocabulary.

**Exercises:** (1) Print which token predicts the first answer token. (2) Compare full/selective/LoRA parameter counts. (3) Change one training answer, rerun from the same seed, and inspect held-out loss. (4) Explain why saving an adapter is not enough to resume AdamW.

**Expected result:** finite objective values, some expected trainable weights changed, frozen weights unchanged, and identical greedy tokens after reload. Record actual values in `lesson_report.json`; no fixed quality threshold is asserted.
